# Exploratory Data Analysis — Mortgage Default Prediction

This notebook explores the cleaned mortgage dataset across multiple years (2000–2008).
The goal is to understand:
1. Class imbalance (default vs non-default)
2. Feature distributions and their evolution over time
3. Early signals of the 2008 financial crisis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load the cleaned dataset
df = pd.read_parquet('../data/03_primary/model_input_cleaned.parquet')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 1. Class Imbalance

Mortgage defaults are rare events — we expect a significant imbalance between
defaulted (1) and non-defaulted (0) loans. This directly affects model choice
and evaluation metrics.

In [ ]:
# Overall default rate
default_counts = df['default'].value_counts()
default_rate = df['default'].mean() * 100

print('=== Overall Class Distribution ===')
print(f'Non-default (0): {default_counts[0]:,} ({100 - default_rate:.2f}%)')
print(f'Default     (1): {default_counts[1]:,} ({default_rate:.2f}%)')
print(f'Imbalance ratio: {default_counts[0] / default_counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Non-default', 'Default'], default_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Loan Count by Default Status')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(default_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    default_counts.values,
    labels=['Non-default', 'Default'],
    colors=['steelblue', 'tomato'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05)
)
axes[1].set_title('Default Rate Distribution')

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Default Rate by Year

The core of the "Big Short" narrative: default rates should be stable in early years
and spike dramatically as we approach 2008.

In [ ]:
# Default rate per origination year
default_by_year = df.groupby('year').agg(
    total_loans=('default', 'count'),
    total_defaults=('default', 'sum'),
    default_rate=('default', 'mean')
).reset_index()
default_by_year['default_rate_pct'] = default_by_year['default_rate'] * 100

print(default_by_year.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default rate over time
axes[0].plot(default_by_year['year'], default_by_year['default_rate_pct'],
             marker='o', color='tomato', linewidth=2.5, markersize=8)
axes[0].axvspan(2005.5, 2008.5, alpha=0.1, color='red', label='Crisis period')
axes[0].set_title('Default Rate by Origination Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Default Rate (%)')
axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

# Loan volume over time
axes[1].bar(default_by_year['year'], default_by_year['total_loans'],
            color='steelblue', edgecolor='white', label='Non-default')
axes[1].bar(default_by_year['year'], default_by_year['total_defaults'],
            color='tomato', edgecolor='white', label='Default')
axes[1].set_title('Loan Volume by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Loans')
axes[1].legend()
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('Default Evolution Over Time (2000–2008)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Key Feature Distributions

Comparing the distribution of key risk indicators between defaulted and non-defaulted loans.

In [ ]:
# Key numeric features to explore
numeric_features = [
    'credit_score',
    'original_ltv',
    'original_dti',
    'original_interest_rate',
    'original_upb',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    if col not in df.columns:
        continue
    # Plot distribution for default=0 and default=1
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = df[df['default'] == label][col].dropna()
        axes[i].hist(subset, bins=50, alpha=0.5, color=color,
                     label='Non-default' if label == 0 else 'Default',
                     density=True)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_ylabel('Density')
    axes[i].legend()

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions: Default vs Non-Default', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Evolution Over Time

Key risk indicators should deteriorate as we approach the crisis.
Rising LTV and DTI ratios, falling credit scores — these are the signals
an MLOps pipeline with drift detection should have caught.

In [ ]:
# Evolution of key risk indicators by year
risk_features = ['credit_score', 'original_ltv', 'original_dti', 'original_interest_rate']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(risk_features):
    if col not in df.columns:
        continue

    yearly_stats = df.groupby('year')[col].agg(['mean', 'median']).reset_index()

    axes[i].plot(yearly_stats['year'], yearly_stats['mean'],
                 marker='o', label='Mean', color='steelblue', linewidth=2)
    axes[i].plot(yearly_stats['year'], yearly_stats['median'],
                 marker='s', label='Median', color='darkorange',
                 linewidth=2, linestyle='--')
    axes[i].axvspan(2005.5, 2008.5, alpha=0.1, color='red', label='Crisis period')
    axes[i].set_title(col.replace('_', ' ').title() + ' Over Time')
    axes[i].set_xlabel('Year')
    axes[i].legend()
    axes[i].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.suptitle('Risk Indicator Evolution (2000–2008)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Missing Values Analysis

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print('=== Missing Values ===')
print(missing_df.to_string())

if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(missing_df.index, missing_df['missing_pct'], color='steelblue')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found.')

## 6. Correlation with Default

Which numeric features are most correlated with the default target?
This helps validate feature selection decisions.

In [ ]:
# Correlation of numeric features with default
numeric_cols = df.select_dtypes(include='number').columns.tolist()
correlations = df[numeric_cols].corr()['default'].drop('default').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['tomato' if v > 0 else 'steelblue' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Default')
ax.set_title('Feature Correlation with Default Target')
plt.tight_layout()
plt.show()

print('\nTop positive correlations (higher = more default risk):')
print(correlations.tail(5).to_string())
print('\nTop negative correlations (lower = less default risk):')
print(correlations.head(5).to_string())

## 7. Train / Validation / Test Split Preview

Visualising the temporal split to confirm class balance across periods.

In [ ]:
# Temporal split: train (2000-2001), val (2002), test (2003-2008)
TRAIN_CUTOFF = 2001
VAL_YEAR = 2002
TEST_START = 2003

train = df[df['year'] <= TRAIN_CUTOFF]
val   = df[df['year'] == VAL_YEAR]
test  = df[df['year'] >= TEST_START]

print('=== Temporal Split Summary ===')
for name, subset in [('Train', train), ('Validation', val), ('Test', test)]:
    print(f"{name:12s}: {len(subset):>7,} loans | "
          f"default rate: {subset['default'].mean()*100:.2f}% | "
          f"years: {subset['year'].min()}–{subset['year'].max()}")

# Visualise split
split_data = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Loans': [len(train), len(val), len(test)],
    'Default Rate (%)': [
        train['default'].mean()*100,
        val['default'].mean()*100,
        test['default'].mean()*100
    ]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['steelblue', 'darkorange', 'tomato']
axes[0].bar(split_data['Split'], split_data['Loans'], color=colors)
axes[0].set_title('Loan Count per Split')
axes[0].set_ylabel('Number of Loans')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].bar(split_data['Split'], split_data['Default Rate (%)'], color=colors)
axes[1].set_title('Default Rate per Split')
axes[1].set_ylabel('Default Rate (%)')

plt.suptitle('Train / Validation / Test Split Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()